# Stage 1 — OSM roads via osmnx

In [ ]:
import osmnx as ox
import geopandas as gpd

# Chennai south bounding box (Adyar river to Velachery)
BBOX = (12.97, 80.19, 13.05, 80.27)  # (south, west, north, east)

# Fetch driveable road network as a NetworkX graph
# G_roads = ox.graph_from_bbox(
#     north=BBOX[2], south=BBOX[0],
#     east=BBOX[3],  west=BBOX[1],
#     network_type="drive",
#     retain_all=False
# )

G_roads = ox.graph_from_bbox(
    bbox=(BBOX[1], BBOX[0], BBOX[3], BBOX[2]),
    custom_filter='["highway"~"primary|secondary|trunk"]',
    retain_all=False
)
# Convert to GeoDataFrame of nodes only (we want centroids, not edges)
nodes_road, _ = ox.graph_to_gdfs(G_roads)
nodes_road = nodes_road[["geometry", "x", "y"]].copy()
nodes_road["node_type"] = "road"
nodes_road = nodes_road.reset_index(drop=True)
print(f"Road nodes: {len(nodes_road)}")


Road nodes: 547


# Stage 2 — OSM power substations via Overpass API

In [ ]:
import requests
import pandas as pd
from shapely.geometry import Point

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

overpass_query = f"""
[out:json][timeout:60];
(
  node["power"="substation"]({BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]});
  way["power"="substation"]({BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]});
  relation["power"="substation"]({BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]});
);
out center;
"""

response = requests.post(OVERPASS_URL, data={"data": overpass_query})
elements = response.json()["elements"]

rows = []
for el in elements:
    if el["type"] == "node":
        lat, lon = el["lat"], el["lon"]
    elif "center" in el:
        lat, lon = el["center"]["lat"], el["center"]["lon"]
    else:
        continue
    rows.append({"x": lon, "y": lat, "geometry": Point(lon, lat), "node_type": "power"})

nodes_power = gpd.GeoDataFrame(rows, crs="EPSG:4326")
print(f"Power substation nodes: {len(nodes_power)}")

Power substation nodes: 11


# Stage 3 — OSM hospitals via osmnx

In [ ]:
tags = {"amenity": ["hospital", "clinic"], "healthcare": "hospital"}

# hospitals_gdf = ox.features_from_bbox(
#     north=BBOX[2], south=BBOX[0],
#     east=BBOX[3],  west=BBOX[1],
#     tags=tags
# )
hospitals_gdf = ox.features_from_bbox(
    bbox=(BBOX[1], BBOX[0], BBOX[3], BBOX[2]),
    tags=tags
)
# Use centroid for polygon geometries (large hospital campuses)
hospitals_gdf = hospitals_gdf[["geometry", "name"]].copy()
hospitals_gdf["geometry"] = hospitals_gdf["geometry"].centroid
hospitals_gdf["x"] = hospitals_gdf.geometry.x
hospitals_gdf["y"] = hospitals_gdf.geometry.y
hospitals_gdf["node_type"] = "hospital"
hospitals_gdf = hospitals_gdf.reset_index(drop=True)
print(f"Hospital nodes: {len(hospitals_gdf)}")

Hospital nodes: 322


/var/folders/mf/_7cg4fl53x78fg3zyp16prcm0000gn/T/ipykernel_33841/1467775069.py:14: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals_gdf["geometry"] = hospitals_gdf["geometry"].centroid


# Stage 4 — SRTM elevation via rasterio

In [ ]:
# Download SRTM tile manually from:
# https://earthdata.nasa.gov  →  search "SRTMGL1"  →  tile N13E080.hgt
# Then load and sample elevation per node coordinate
import elevation
import rasterio
import numpy as np

# SRTM_PATH = "N13E080.hgt"  # place in working directory after download
# Download the SRTM 30m tile covering Chennai automatically
elevation.clip(bounds=(80.19, 12.97, 80.27, 13.05), output="chennai_dem.tif")
elevation.clean()  # clean up temp files
SRTM_PATH = "chennai_dem.tif"
def sample_elevation(gdf, srtm_path):
    """Add an 'elevation_m' column to gdf by sampling the SRTM raster."""
    with rasterio.open(srtm_path) as src:
        coords = list(zip(gdf["x"], gdf["y"]))
        elevations = [v[0] for v in src.sample(coords)]
    gdf = gdf.copy()
    gdf["elevation_m"] = elevations
    gdf["elevation_m"] = gdf["elevation_m"].clip(lower=0)  # no-data → 0
    return gdf

make: Nothing to be done for `download'.
make: Nothing to be done for `all'.
cp SRTM1.vrt SRTM1.845ac5edcb7243859835a8d6cb6122d2.vrt
gdal_translate -q -co TILED=YES -co COMPRESS=DEFLATE -co ZLEVEL=9 -co PREDICTOR=2 -projwin 80.19 13.05 80.27 12.97 SRTM1.845ac5edcb7243859835a8d6cb6122d2.vrt chennai_dem.tif


rm -f SRTM1.845ac5edcb7243859835a8d6cb6122d2.vrt
find cache -size 0 -name "*.tif" -delete
rm -f SRTM1.*.vrt
rm -f -r spool/*


# Stage 5 — Merge all nodes and build the NetworkX graph

In [ ]:
import networkx as nx
from scipy.spatial import cKDTree

import srtm
import numpy as np
# 5a. Merge
all_nodes = pd.concat(
    [nodes_road, nodes_power, hospitals_gdf[["geometry","x","y","node_type"]]],
    ignore_index=True
)
all_nodes = gpd.GeoDataFrame(all_nodes, crs="EPSG:4326")

# 5b. Project to metric CRS for distance calculations
# all_nodes_m = all_nodes.to_crs("EPSG:32644")
# all_nodes_m["mx"] = all_nodes_m.geometry.x
# all_nodes_m["my"] = all_nodes_m.geometry.y
all_nodes_m = all_nodes.to_crs("EPSG:32644")
all_nodes_m["mx"] = all_nodes_m.geometry.x
all_nodes_m["my"] = all_nodes_m.geometry.y
# 5c. Sample elevation for every node
# all_nodes_m = sample_elevation(all_nodes_m.to_crs("EPSG:4326"), SRTM_PATH)
# all_nodes_m = all_nodes_m.to_crs("EPSG:32644")  # back to metric
print("Fetching SRTM elevation data... (downloads tiles on first run, cached after)")
srtm_data = srtm.get_data()
all_nodes_wgs = all_nodes_m.to_crs("EPSG:4326")

elevations = []
for _, row in all_nodes_wgs.iterrows():
    elev = srtm_data.get_elevation(row.geometry.y, row.geometry.x)
    elevations.append(float(elev) if elev is not None else 0.0)

all_nodes_m = all_nodes_m.copy()
all_nodes_m["elevation_m"] = np.clip(elevations, 0, None)

print(f"Elevation stats:\n{all_nodes_m['elevation_m'].describe()}")

# 5d. Build graph
G = nx.Graph()
coords_arr = np.array(list(zip(all_nodes_m["mx"], all_nodes_m["my"])))
tree = cKDTree(coords_arr)

for i, row in all_nodes_m.iterrows():
    G.add_node(i,
        node_type=row["node_type"],
        x=row["x"], y=row["y"],
        elevation_m=row["elevation_m"]
    )

# Edge rules (distances in metres)
RULES = {
    ("road",    "road"):     500,   # same road network proximity
    ("hospital", "power"): 2000,  # substation powers nearby hospitals
    ("hospital", "road"): 500,   # hospital reachable via nearest road
    ("power",    "road"):    1000,  # substation needs road for maintenance
}

type_arr = all_nodes_m["node_type"].values
for i in range(len(all_nodes_m)):
    t_i = type_arr[i]
    max_radius = max(r for (a,b),r in RULES.items() if t_i in (a,b))
    candidates = tree.query_ball_point(coords_arr[i], r=max_radius)
    for j in candidates:
        if j <= i:
            continue
        t_j = type_arr[j]
        pair = tuple(sorted([t_i, t_j]))
        if pair in RULES:
            dist = np.linalg.norm(coords_arr[i] - coords_arr[j])
            if dist <= RULES[pair]:
                G.add_edge(i, j, weight=dist, edge_type=f"{pair[0]}-{pair[1]}")

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Fetching SRTM elevation data... (downloads tiles on first run, cached after)
Elevation stats:
count    880.000000
mean      10.936364
std        3.459901
min        0.000000
25%        9.000000
50%       11.000000
75%       13.000000
max       24.000000
Name: elevation_m, dtype: float64
Graph: 880 nodes, 6792 edges


# CHECK

In [ ]:
print(all_nodes["node_type"].value_counts())

print(all_nodes_m["elevation_m"].describe())

print(nx.number_connected_components(G), "connected components")

node_type
road        547
hospital    322
power        11
Name: count, dtype: int64
count    880.000000
mean      10.936364
std        3.459901
min        0.000000
25%        9.000000
50%       11.000000
75%       13.000000
max       24.000000
Name: elevation_m, dtype: float64
13 connected components


# Stage 6 — Power grid (expanded Overpass query)

In [ ]:
# Stage 6 — Full power grid: substations + lines + towers
import requests
import geopandas as gpd
from shapely.geometry import Point
import json

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

power_query = f"""
[out:json][timeout:90];
(
  node["power"="substation"]({BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]});
  way["power"="substation"]({BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]});
  node["power"="tower"]({BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]});
  node["power"="pole"]({BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]});
  way["power"="line"]({BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]});
  way["power"="minor_line"]({BBOX[0]},{BBOX[1]},{BBOX[2]},{BBOX[3]});
);
out center;
"""

import json
with open('power_grid_raw.json', 'r') as f:

    power_elements = json.load(f)['elements']
print('Loaded from power_grid_raw.json')
print(f"Raw power JSON saved → power_grid_raw.json")

rows = []
for el in power_elements:
    if el["type"] == "node":
        lat, lon = el["lat"], el["lon"]
    elif "center" in el:
        lat, lon = el["center"]["lat"], el["center"]["lon"]
    else:
        continue
    power_type = el.get("tags", {}).get("power", "unknown")
    rows.append({
        "x": lon, "y": lat,
        "geometry": Point(lon, lat),
        "node_type": "power",
        "power_subtype": power_type,   # substation / tower / pole / line
        "osm_id": el["id"]
    })

nodes_power_full = gpd.GeoDataFrame(rows, crs="EPSG:4326")
nodes_power_full.to_file("power_grid_nodes.gpkg", driver="GPKG")

print(f"Power grid nodes: {len(nodes_power_full)}")
print(nodes_power_full["power_subtype"].value_counts())

Loaded from power_grid_raw.json
Raw power JSON saved → power_grid_raw.json
Power grid nodes: 43
power_subtype
tower         25
substation    11
line           6
pole           1
Name: count, dtype: int64


# Stage 7 — Flood extent raster (synthetic from SRTM + flood model)


In [ ]:
# Stage 7 — Synthetic flood extent raster from SRTM DEM
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
import geopandas as gpd
from shapely.geometry import box

# ── 7a. Re-sample SRTM elevation onto a regular grid ──────────────────────
# Grid resolution: 100m cells over your bounding box
GRID_RES = 0.001  # ~100m in degrees

lons = np.arange(BBOX[1], BBOX[3], GRID_RES)
lats = np.arange(BBOX[0], BBOX[2], GRID_RES)
lon_grid, lat_grid = np.meshgrid(lons, lats)

print("Sampling elevation grid for flood model...")
elev_grid = np.zeros_like(lon_grid, dtype=np.float32)
for i in range(lat_grid.shape[0]):
    for j in range(lon_grid.shape[1]):
        e = srtm_data.get_elevation(float(lat_grid[i, j]), float(lon_grid[i, j]))
        elev_grid[i, j] = float(e) if e is not None else 0.0

elev_grid = np.clip(elev_grid, 0, None)
print(f"Elevation grid shape: {elev_grid.shape}, range: {elev_grid.min():.1f}–{elev_grid.max():.1f}m")

# ── 7b. Bathtub flood model: flood depth = max(0, flood_level - elevation) ─
# Chennai 2015: Adyar river overflowed, flood level ~3–5m in low areas
# We parameterise by flood_level (metres above sea level)
FLOOD_LEVEL_M = 5.0   # tune this — 5m replicates severe 2015 scenario

flood_depth = np.maximum(0.0, FLOOD_LEVEL_M - elev_grid).astype(np.float32)
flooded_cells = np.sum(flood_depth > 0)
print(f"Flooded cells at {FLOOD_LEVEL_M}m level: {flooded_cells} / {flood_depth.size} "
      f"({100*flooded_cells/flood_depth.size:.1f}%)")

# ── 7c. Save flood depth as GeoTIFF ───────────────────────────────────────
transform = from_bounds(
    west=BBOX[1], south=BBOX[0],
    east=BBOX[3], north=BBOX[2],
    width=lon_grid.shape[1], height=lat_grid.shape[0]
)

with rasterio.open(
    "chennai_flood_depth.tif",
    "w",
    driver="GTiff",
    height=flood_depth.shape[0],
    width=flood_depth.shape[1],
    count=1,
    dtype=np.float32,
    crs=CRS.from_epsg(4326),
    transform=transform,
    nodata=-9999.0
) as dst:
    dst.write(flood_depth, 1)

print("Flood depth raster saved → chennai_flood_depth.tif")
print(f"  Flood level used: {FLOOD_LEVEL_M}m above sea level")
print(f"  Max depth: {flood_depth.max():.2f}m")
print(f"  Mean depth (flooded only): {flood_depth[flood_depth>0].mean():.2f}m")

# ── 7d. Also save flood extent as vector polygons (for Folium visualisation)
from rasterio.features import shapes
from shapely.geometry import shape

mask = (flood_depth > 0).astype(np.uint8)
flood_polys = []
for geom, val in shapes(mask, transform=transform):
    if val == 1:
        flood_polys.append(shape(geom))

flood_extent_gdf = gpd.GeoDataFrame(geometry=flood_polys, crs="EPSG:4326")
flood_extent_gdf["flood_level_m"] = FLOOD_LEVEL_M
flood_extent_gdf.to_file("chennai_flood_extent.gpkg", driver="GPKG")
print(f"Flood extent vector saved → chennai_flood_extent.gpkg ({len(flood_polys)} polygons)")

Sampling elevation grid for flood model...


Elevation grid shape: (81, 80), range: 0.0–72.0m
Flooded cells at 5.0m level: 655 / 6480 (10.1%)
Flood depth raster saved → chennai_flood_depth.tif
  Flood level used: 5.0m above sea level
  Max depth: 5.00m
  Mean depth (flooded only): 2.51m
Flood extent vector saved → chennai_flood_extent.gpkg (48 polygons)


# Stage 8 — Add flood depth to each node + save final graph

In [ ]:
# Stage 8 — Attach flood depth to nodes and save the complete graph

import networkx as nx
import pickle
import json
import numpy as np
import rasterio

# ── 8a. Sample flood depth at every node location ─────────────────────────
with rasterio.open("chennai_flood_depth.tif") as flood_src:
    node_coords_wgs = [(data["x"], data["y"]) for _, data in G.nodes(data=True)]
    flood_depths = [v[0] for v in flood_src.sample(node_coords_wgs)]

for (node_id, data), depth in zip(G.nodes(data=True), flood_depths):
    G.nodes[node_id]["flood_depth_m"] = float(max(0.0, depth))

print("Flood depth attached to all graph nodes.")
print(f"  Flooded nodes (depth > 0): "
      f"{sum(1 for _,d in G.nodes(data=True) if d['flood_depth_m'] > 0)}")

# ── 8b. Prune to largest connected component ──────────────────────────────
largest_cc = max(nx.connected_components(G), key=len)
G_pruned = G.subgraph(largest_cc).copy()
print(f"After pruning to largest component: "
      f"{G_pruned.number_of_nodes()} nodes, {G_pruned.number_of_edges()} edges")

# ── 8c. Save graph in multiple formats ────────────────────────────────────

# Format 1: pickle (fastest, preserves all Python objects)
with open("cascadewatch_graph.pkl", "wb") as f:
    pickle.dump(G_pruned, f)
print("Graph saved → cascadewatch_graph.pkl")

# Format 2: GraphML (human-readable XML, loads in Gephi/Cytoscape)
nx.write_graphml(G_pruned, "cascadewatch_graph.graphml")
print("Graph saved → cascadewatch_graph.graphml")

# Format 3: node/edge CSV (for inspection in pandas/Excel)
import pandas as pd

node_records = [
    {"node_id": n, **d}
    for n, d in G_pruned.nodes(data=True)
]
edge_records = [
    {"src": u, "dst": v, **d}
    for u, v, d in G_pruned.edges(data=True)
]

pd.DataFrame(node_records).to_csv("cascadewatch_nodes.csv", index=False)
pd.DataFrame(edge_records).to_csv("cascadewatch_edges.csv", index=False)
print("Nodes saved → cascadewatch_nodes.csv")
print("Edges saved → cascadewatch_edges.csv")

# ── 8d. Summary ───────────────────────────────────────────────────────────
node_df = pd.DataFrame(node_records)
print("\n── Final graph summary ──────────────────────────────────")
print(node_df["node_type"].value_counts().to_string())
print(f"\nElevation  — mean: {node_df['elevation_m'].mean():.1f}m  "
      f"max: {node_df['elevation_m'].max():.1f}m")
print(f"Flood depth — mean: {node_df['flood_depth_m'].mean():.2f}m  "
      f"max: {node_df['flood_depth_m'].max():.2f}m")
print(f"Connected components in pruned graph: "
      f"{nx.number_connected_components(G_pruned)}")
print("─────────────────────────────────────────────────────────")

Flood depth attached to all graph nodes.
  Flooded nodes (depth > 0): 156
After pruning to largest component: 820 nodes, 6652 edges
Graph saved → cascadewatch_graph.pkl


Graph saved → cascadewatch_graph.graphml
Nodes saved → cascadewatch_nodes.csv
Edges saved → cascadewatch_edges.csv

── Final graph summary ──────────────────────────────────
node_type
road        511
hospital    298
power        11

Elevation  — mean: 10.8m  max: 24.0m
Flood depth — mean: 0.51m  max: 5.00m
Connected components in pruned graph: 1
─────────────────────────────────────────────────────────
